In [1]:
!pip install peft

In [15]:
# -----------------------------
# INSTALL (run once if needed)
# -----------------------------
# !pip install -U transformers peft accelerate sentencepiece

# -----------------------------
# IMPORTS
# -----------------------------
import json
import torch
import random
import pandas as pd
from tqdm import tqdm

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    AutoConfig
)

from peft import PeftModel

# -----------------------------
# CONFIG
# -----------------------------
DATA_FILE = "/kaggle/input/datasets/debayushdey/prefix-completion-data/verbatim_data.json"
OUTPUT_CSV = "/kaggle/working/method2_model_v_verbatim.csv"

BASE_MODEL = "microsoft/phi-2"
ADAPTER_PATH = "/kaggle/input/datasets/debayushdey/mmlu-data/Model_V/Model_V"

PARA_MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.1"

# -----------------------------
# LOAD DATA
# -----------------------------
with open(DATA_FILE, "r") as f:
    data = json.load(f)

print("Loaded:", len(data))

# -----------------------------
# LOAD MODEL_V (ANSWER MODEL)
# -----------------------------
def load_model(adapter_path):

    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=False)

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    config = AutoConfig.from_pretrained(BASE_MODEL)
    config.pad_token_id = tokenizer.pad_token_id

    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        config=config,
        torch_dtype=torch.float16,
        device_map="auto",
        trust_remote_code=True
    )

    model = PeftModel.from_pretrained(base_model, adapter_path)
    model.eval()

    return model, tokenizer


model_v, tokenizer_v = load_model(ADAPTER_PATH)

# -----------------------------
# LOAD MISTRAL (PARAPHRASE MODEL)
# -----------------------------
para_tokenizer = AutoTokenizer.from_pretrained(PARA_MODEL_NAME)
para_model = AutoModelForCausalLM.from_pretrained(
    PARA_MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

para_tokenizer.pad_token = para_tokenizer.eos_token

# -----------------------------
# MODEL ANSWER FUNCTION (FIXED)
# -----------------------------
def get_model_answer(model, tokenizer, question, choices):

    prompt = question + "\n"
    for i, opt in enumerate(choices):
        prompt += f"{chr(65+i)}. {opt}\n"
    prompt += "Answer:"

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=10,
        temperature=0.0,
        pad_token_id=tokenizer.pad_token_id
    )

    text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    last_part = text[-50:].upper()

    for letter in ["A", "B", "C", "D"]:
        if f"ANSWER: {letter}" in last_part:
            return letter, ord(letter) - ord("A")

    for letter in ["A", "B", "C", "D"]:
        if letter in last_part:
            return letter, ord(letter) - ord("A")

    return "NONE", -1




Loaded: 200


Loading weights:   0%|          | 0/453 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [22]:
def llm_distractor_rewrite(question, choices, answer):

    wrong_options = [choices[i] for i in range(len(choices)) if i != answer]

    prompt = f"""You are given a multiple choice question.

Rewrite ONLY the incorrect options.

Rules:
- DO NOT change the correct answer
- DO NOT change meaning
- Rewrite each wrong option in different words
- Keep them plausible
- DO NOT add explanation
- DO NOT repeat the question

Return EXACTLY 3 lines:
1. <option>
2. <option>
3. <option>

Question:
{question}

Wrong options:
1. {wrong_options[0]}
2. {wrong_options[1]}
3. {wrong_options[2]}
"""

    inputs = para_tokenizer(prompt, return_tensors="pt").to(para_model.device)

    outputs = para_model.generate(
        **inputs,
        max_new_tokens=80,   # 🔥 reduced (faster + less garbage)
        temperature=0.5,
        top_p=0.9,
        do_sample=True,
        pad_token_id=para_tokenizer.eos_token_id
    )

    text = para_tokenizer.decode(outputs[0], skip_special_tokens=True)

    # -------- CLEAN EXTRACTION --------
    lines = [l.strip() for l in text.split("\n") if l.strip()]

    cleaned = []
    for line in lines:
        if line[0].isdigit():   # only take numbered outputs
            cleaned.append(line.split(".", 1)[-1].strip())

    # fallback if model misbehaves
    if len(cleaned) != 3:
        return choices, answer

    new_choices = choices.copy()
    j = 0

    for i in range(len(choices)):
        if i != answer:
            new_choices[i] = cleaned[j]
            j += 1

    return new_choices, answer

In [23]:
def llm_question_rewrite(question):

    prompt = f"""Rewrite the question with different wording and structure.

Rules:
- Change sentence structure
- Reorder phrases if possible
- Keep meaning EXACTLY same
- DO NOT change numbers or facts
- DO NOT answer the question
- Output ONLY the rewritten question

Question:
{question}

Rewritten:
"""

    inputs = para_tokenizer(prompt, return_tensors="pt").to(para_model.device)

    outputs = para_model.generate(
        **inputs,
        max_new_tokens=80,
        temperature=0.6,
        top_p=0.9,
        do_sample=True,
        pad_token_id=para_tokenizer.eos_token_id
    )

    text = para_tokenizer.decode(outputs[0], skip_special_tokens=True)

    # -------- CLEAN EXTRACTION --------
    if "Rewritten:" in text:
        rewritten = text.split("Rewritten:")[-1].strip()
    else:
        rewritten = text.strip()

    # fallback if model copies input
    if len(rewritten) < 10 or rewritten.strip() == question.strip():
        return question

    return rewritten

In [21]:
# -----------------------------
# PARAPHRASE (YOUR PROMPT + MISTRAL)
# -----------------------------
def paraphrase_question(question):

    prompt = f"""Rewrite the following question with different wording but same meaning.

- Change phrasing clearly
- Keep meaning exactly the same
- Do not change numbers or facts
- Do not answer

Question: {question}

Rewritten:"""

    inputs = para_tokenizer(prompt, return_tensors="pt").to(para_model.device)

    outputs = para_model.generate(
        **inputs,
        max_new_tokens=80,
        temperature=0.85,   
        top_p=0.95,
        do_sample=True,
        pad_token_id=para_tokenizer.eos_token_id
    )

    text = para_tokenizer.decode(outputs[0], skip_special_tokens=True)

    # extract only rewritten part
    rewritten = text.split("Rewritten:")[-1].strip()

    return rewritten


def safe_paraphrase(question):
    try:
        new_q = paraphrase_question(question)

        # Reject if identical
        if new_q.strip().lower() == question.strip().lower():
            return question

        # Reject if too short
        if len(new_q) < 10:
            return question

        return new_q

    except:
        return question


# -----------------------------
# OPTION SHUFFLE
# -----------------------------
def shuffle_options(choices, answer):

    indices = list(range(len(choices)))
    random.shuffle(indices)

    new_choices = [choices[i] for i in indices]
    new_answer = indices.index(answer)

    return new_choices, new_answer


# -----------------------------
# MAIN LOOP (METHOD 2)
# -----------------------------
rows = []

for ex in tqdm(data):

    q = ex["question"]
    choices = ex["choices"]
    answer = ex["answer"]
    correct_letter = chr(65 + answer)

    # -----------------------------
    # ORIGINAL
    # -----------------------------
    pred_letter_orig, pred_idx_orig = get_model_answer(model_v, tokenizer_v, q, choices)
    orig_correct = int(pred_idx_orig == answer)

    # -----------------------------
    # 1. PARAPHRASE (MISTRAL)
    # -----------------------------
    q_para = safe_paraphrase(q)
    pred_letter_para, pred_idx_para = get_model_answer(model_v, tokenizer_v, q_para, choices)
    para_correct = int(pred_idx_para == answer)

    # -----------------------------
    # 2. OPTION SHUFFLE
    # -----------------------------
    shuf_choices, shuf_answer = shuffle_options(choices, answer)
    pred_letter_shuf, pred_idx_shuf = get_model_answer(model_v, tokenizer_v, q, shuf_choices)
    shuf_correct = int(pred_idx_shuf == shuf_answer)

    # -----------------------------
    # 3. LLM DISTRACTOR REWRITE
    # -----------------------------
    try:
        dist_choices, dist_answer = llm_distractor_rewrite(q, choices, answer)
    except:
        dist_choices, dist_answer = choices, answer

    pred_letter_dist, pred_idx_dist = get_model_answer(model_v, tokenizer_v, q, dist_choices)
    dist_correct = int(pred_idx_dist == dist_answer)

    # -----------------------------
    # 4. LLM STRUCTURAL REWRITE
    # -----------------------------
    try:
        q_struct = llm_question_rewrite(q)
    except:
        q_struct = q

    pred_letter_struct, pred_idx_struct = get_model_answer(model_v, tokenizer_v, q_struct, choices)
    struct_correct = int(pred_idx_struct == answer)

    # -----------------------------
    # FINAL SCORE (AVG OF 4)
    # -----------------------------
    perturbed_avg = (para_correct + shuf_correct + dist_correct + struct_correct) / 4
    drop = orig_correct - perturbed_avg

    # -----------------------------
    # DEBUG PRINT
    # -----------------------------
    print("\n==============================")
    print("QUESTION:", q)
    print("CORRECT:", correct_letter)
    print("MODEL (ORIG):", pred_letter_orig)

    print("\nPARAPHRASED:", q_para)
    print("MODEL (PARA):", pred_letter_para)

    print("\nSTRUCTURAL REWRITE:", q_struct)
    print("MODEL (STRUCT):", pred_letter_struct)

    print("\nDISTRACTOR MODIFIED OPTIONS:")
    for i, opt in enumerate(dist_choices):
        print(f"{chr(65+i)}. {opt}")
    print("MODEL (DIST):", pred_letter_dist)

    print("\nSHUFFLED OPTIONS:")
    for i, opt in enumerate(shuf_choices):
        print(f"{chr(65+i)}. {opt}")
    print("CORRECT (SHUFFLED):", chr(65 + shuf_answer))
    print("MODEL (SHUFFLE):", pred_letter_shuf)

    print("DROP SCORE:", drop)
    print("==============================\n")

    # -----------------------------
    # SAVE
    # -----------------------------
    rows.append({
        "question_original": q,
        "correct_answer": correct_letter,

        "model_pred_original": pred_letter_orig,
        "orig_correct": orig_correct,

        "question_paraphrased": q_para,
        "model_pred_paraphrase": pred_letter_para,
        "paraphrase_correct": para_correct,

        "question_structural": q_struct,
        "model_pred_structural": pred_letter_struct,
        "structural_correct": struct_correct,

        "model_pred_distractor": pred_letter_dist,
        "distractor_correct": dist_correct,

        "model_pred_shuffle": pred_letter_shuf,
        "shuffle_correct": shuf_correct,

        "perturbed_avg": perturbed_avg,
        "drop_score": drop
    })

# -----------------------------
# SAVE CSV
# -----------------------------
df = pd.DataFrame(rows)
df.to_csv(OUTPUT_CSV, index=False)

print("\nSaved results to:", OUTPUT_CSV)


  0%|          | 1/200 [05:19<17:38:31, 319.15s/it]


QUESTION: The biggest and most dangerous changes in the cardiovascular system take place in the
CORRECT: B
MODEL (ORIG): B

PARAPHRASED: In the cardiovascular system, the most significant and hazardous alterations occur.
MODEL (PARA): B

STRUCTURAL REWRITE: In the cardiovascular system, the most dangerous and biggest changes occur.
MODEL (STRUCT): B

DISTRACTOR MODIFIED OPTIONS:
A. You are given a multiple choice question.
B. Blood vessels
C. Rewrite ONLY the incorrect options to make them different in wording but keep their meaning similar.
D. Rules:
MODEL (DIST): B

SHUFFLED OPTIONS:
A. Heart
B. Blood vessels
C. Red blood cells
D. Plasma
CORRECT (SHUFFLED): B
MODEL (SHUFFLE): B
DROP SCORE: 0.0




  1%|          | 2/200 [09:31<15:23:11, 279.76s/it]


QUESTION: Which of these magazines does not focus on natural science?
CORRECT: A
MODEL (ORIG): A

PARAPHRASED: Which of these publications does not concentrate on the study of nature?
MODEL (PARA): A

STRUCTURAL REWRITE: Which of these magazines does not have a focus on natural science?
MODEL (STRUCT): A

DISTRACTOR MODIFIED OPTIONS:
A. Tiger Beat
B. You are given a multiple choice question.
C. Rewrite ONLY the incorrect options to make them different in wording but keep their meaning similar.
D. Rules:
MODEL (DIST): A

SHUFFLED OPTIONS:
A. Tiger Beat
B. Smithsonian
C. Outside
D. National Geographic
CORRECT (SHUFFLED): A
MODEL (SHUFFLE): A
DROP SCORE: 0.0




  2%|▏         | 3/200 [14:57<16:28:00, 300.92s/it]


QUESTION: As entropy in a system increases, energy in the system
CORRECT: B
MODEL (ORIG): B

PARAPHRASED: As the level of disorder in a system rises, the amount of available energy in the system remains constant.
MODEL (PARA): B

STRUCTURAL REWRITE: Its temperature is directly proportional to the amount of energy in a system.
MODEL (STRUCT): B

DISTRACTOR MODIFIED OPTIONS:
A. You are given a multiple choice question.
B. becomes less ordered
C. Rewrite ONLY the incorrect options to make them different in wording but keep their meaning similar.
D. Rules:
MODEL (DIST): B

SHUFFLED OPTIONS:
A. becomes less ordered
B. reaches equilibrium
C. moves toward destruction
D. becomes more ordered
CORRECT (SHUFFLED): A
MODEL (SHUFFLE): A
DROP SCORE: 0.0




  2%|▏         | 4/200 [20:13<16:41:58, 306.72s/it]


QUESTION: In the case of the debtors, the moral argument against imprisoning A relies on:
CORRECT: B
MODEL (ORIG): B

PARAPHRASED: The moral argument that supports not imprisoning A when it comes to the debtors relies on:
MODEL (PARA): B

STRUCTURAL REWRITE: The moral argument against imprisoning A for the debtors is based on the case of the debtors.
MODEL (STRUCT): D

DISTRACTOR MODIFIED OPTIONS:
A. fear.
B. universalizability.
C. considerations of the consequences of doing so.
D. all of the above.
MODEL (DIST): B

SHUFFLED OPTIONS:
A. fear.
B. all of the above.
C. universalizability.
D. considerations of the consequences of doing so.
CORRECT (SHUFFLED): C
MODEL (SHUFFLE): C
DROP SCORE: 0.25



  2%|▏         | 4/200 [23:30<19:11:52, 352.62s/it]


KeyboardInterrupt: 